# AI-Publisher Model Test
ToS uyumlu: direkt HuggingFace inference, Docker/Flask/Ngrok yok, kisa session.

In [ ]:
# === HAZIRLIK: Drive mount + bagimliliklar ===
import os, gc, time, json
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

os.environ['HF_HOME'] = '/content/drive/MyDrive/Colab_Cache/huggingface'
os.environ['TORCH_HOME'] = '/content/drive/MyDrive/Colab_Cache/torch'
os.environ['XDG_CACHE_HOME'] = '/content/drive/MyDrive/Colab_Cache'
os.makedirs('/content/drive/MyDrive/Colab_Cache', exist_ok=True)

OUTPUT_DIR = '/content/output'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab_Notebooks/test_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

results_log = []

!pip install -q diffusers transformers accelerate imageio[ffmpeg] torch opencv-python-headless scipy
import torch; import numpy as np; import imageio

In [ ]:
# === edge-tts kurulumu (Colab test, XTTS RunPod'da) ===
!pip install -q edge-tts

In [ ]:
# === GPU KONTROL ===
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
total = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'VRAM: {total:.1f} GB')

In [ ]:
def write_video_ffmpeg(frames, out_path, fps=8):
    """Write frames to playable mp4 via ffmpeg subprocess."""
    import subprocess as sp, numpy as np, os
    first = np.array(frames[0])
    if first.dtype != np.uint8:
        frames = [(np.clip(np.array(f), 0.0, 1.0) * 255).astype(np.uint8) for f in frames]
    else:
        frames = [np.array(f) for f in frames]
    h, w = frames[0].shape[:2]
    cmd = [
        'ffmpeg', '-y',
        '-f', 'rawvideo', '-vcodec', 'rawvideo',
        '-s', f'{w}x{h}', '-pix_fmt', 'rgb24', '-r', str(fps),
        '-i', '-',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        '-preset', 'fast', '-crf', '18',
        out_path
    ]
    proc = sp.Popen(cmd, stdin=sp.PIPE)
    for f in frames:
        proc.stdin.write(f.tobytes())
    proc.stdin.close()
    proc.wait()
    return out_path
# === HELPER: Run Test ===
def run_test(name, run_fn):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    vram_before = torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 0
    start = time.time()
    result = {'name': name, 'status': 'FAIL', 'duration_s': 0, 'vram_used_gb': 0, 'file_size_mb': 0, 'file_path': '', 'error': ''}
    try:
        out_paths = run_fn()
        elapsed = time.time() - start
        vram_after = torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 0
        result['status'] = 'OK'
        result['duration_s'] = round(elapsed, 1)
        result['vram_used_gb'] = round(vram_before - vram_after, 2)
        if out_paths:
            for p in out_paths:
                if os.path.exists(p):
                    result['file_path'] = p
                    result['file_size_mb'] = round(os.path.getsize(p) / (1024*1024), 1)
                    fname = os.path.basename(p)
                    import shutil
                    shutil.copy2(p, os.path.join(DRIVE_OUTPUT_DIR, fname))
                    print(f'[COPY] Drive: {DRIVE_OUTPUT_DIR}/{fname}')
    except Exception as e:
        result['error'] = str(e); print(f'[ERROR] {e}')
    finally:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        results_log.append(result)
        print(f'[DONE] {name}: {result["status"]} ({result["duration_s"]}s)')
        print('---')
    return result

---
## Test 1: Gorsel Uretimi (SDXL)

In [ ]:
def test_sdxl():
    from diffusers import StableDiffusionXLPipeline
    print('[LOAD] SDXL yukleniyor...')
    pipe = StableDiffusionXLPipeline.from_pretrained(
        'stabilityai/stable-diffusion-xl-base-1.0',
        torch_dtype=torch.float16, variant='fp16', use_safetensors=True
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
    prompt = 'A photorealistic close-up of a steaming coffee cup on a rustic wooden table, soft morning sunlight streaming through a window, a single white daisy in a small vase beside the cup, warm amber tones, shallow depth of field, ultra-detailed, 8K'
    print(f'[RUN] prompt: {prompt}')
    image = pipe(prompt=prompt, num_inference_steps=25).images[0]
    out = f'{OUTPUT_DIR}/sdxl_test.png'
    image.save(out); display(image)
    del pipe; return [out]

run_test('SDXL (Gorsel)', test_sdxl)

---
## Test 2: Video Uretimi (CogVideoX-2b)

In [ ]:
def test_cogvideo():
    from diffusers import CogVideoXPipeline
    print('[LOAD] CogVideoX-2b yukleniyor...')
    pipe = CogVideoXPipeline.from_pretrained('THUDM/CogVideoX-2b', torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    prompt = 'Cinematic sunset over a calm Mediterranean sea, golden orange and pink sky reflecting on gentle rippling water, waves softly kissing a sandy shore, a lone sailboat silhouetted on the horizon, warm nostalgic atmosphere, smooth camera panning right, 24fps cinematic quality'
    print(f'[RUN] prompt: {prompt}')
    output = pipe(prompt=prompt, num_frames=25, num_inference_steps=25, guidance_scale=6)
    frames = output.frames[0]
    out = f'{OUTPUT_DIR}/cogvideo_test.mp4'
    write_video_ffmpeg(frames, out, fps=8)
    from IPython.display import Video
    display(Video(out, width=512))
    del pipe; return [out]

run_test('CogVideoX-2b (Video)', test_cogvideo)

---
## Test 3: Gorselden Video (CogVideoX-2b-I2V)

In [ ]:
def test_i2v():
    init_img = f'{OUTPUT_DIR}/sdxl_test.png'
    if not os.path.exists(init_img):
        print('[SKIP] Test 1 gorseli bulunamadi'); return []
    from diffusers import StableVideoDiffusionPipeline
    from diffusers.utils import load_image
import cv2
    print('[LOAD] SVD (Stable Video Diffusion) yukleniyor...')
    pipe = StableVideoDiffusionPipeline.from_pretrained('stabilityai/stable-video-diffusion-img2vid', torch_dtype=torch.float16, variant='fp16')
    pipe.enable_model_cpu_offload()
    image = load_image(init_img)
    image = image.resize((1024, 576))
    print('[RUN] I2V: kahve gorselinden video...')
    frames = pipe(image=image, decode_chunk_size=8, num_frames=25).frames[0]
    out = f'{OUTPUT_DIR}/svd_i2v_test.mp4'
    h, w = frames[0].size[1], frames[0].size[0]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw = cv2.VideoWriter(out, fourcc, 7, (w, h))
    for f in frames:
        vw.write(cv2.cvtColor(np.array(f), cv2.COLOR_RGB2BGR))
    vw.release()
    from IPython.display import Video
    display(Video(out, width=512))
    del pipe; return [out]

run_test('SVD (Gorselden Video)', test_i2v)

---
## Test 4: Turkce TTS (XTTS)

In [ ]:
def test_tts():
    # edge-tts: Microsoft Edge TTS API, basit bagimlilik, Turkce destekler
    # XTTS RunPod'da test edilecek
    !pip install -q edge-tts
    import edge_tts, asyncio
    text = 'AI Publisher video uretim sistemine hos geldiniz. Bugun sizlere ozel bir icerik hazirladik.'
    print(f'[RUN] text: {text}')
    out = f'{OUTPUT_DIR}/tts_test.mp3'
    async def _run():
        communicate = edge_tts.Communicate(text, voice='tr-TR-EmelNeural')
        await communicate.save(out)
    asyncio.run(_run())
    from IPython.display import Audio
    display(Audio(out))
    return [out]

run_test('Edge-TTS (Turkce Ses)', test_tts)

---
## Test 5: Ses Efekti (AudioLDM2)

In [ ]:
def test_sfx():
    from diffusers import AudioLDM2Pipeline
    print('[LOAD] AudioLDM2 yukleniyor...')
    import json, os
    from huggingface_hub import snapshot_download
    model_path = snapshot_download(repo_id='cvssp/audioldm2')
    # Patch model_index.json AND text_encoder/config.json
    for rel_path in ['model_index.json', os.path.join('text_encoder', 'config.json')]:
        cfg_path = os.path.join(model_path, rel_path)
        if os.path.exists(cfg_path):
            with open(cfg_path) as f:
                cfg = f.read()
            if 'GPT2Model' in cfg:
                with open(cfg_path, 'w') as f:
                    f.write(cfg.replace('GPT2Model', 'GPT2LMHeadModel'))
                print(f'[PATCH] {rel_path}: GPT2Model -> GPT2LMHeadModel')
    pipe = AudioLDM2Pipeline.from_pretrained('cvssp/audioldm2', torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    prompt = 'Gentle ocean waves rhythmically crashing on a sandy beach, seagulls calling in the distance, light breeze rustling palm leaves, peaceful and relaxing coastal atmosphere, high quality ambient field recording'
    print(f'[RUN] prompt: {prompt}')
    audio = pipe(prompt=prompt, audio_length_in_s=6.0, num_inference_steps=20).audios[0]
    from scipy.io.wavfile import write as write_wav
    out = f'{OUTPUT_DIR}/sfx_test.wav'
    write_wav(out, 16000, audio)
    from IPython.display import Audio
    display(Audio(out))
    del pipe; return [out]

run_test('AudioLDM2 (Ses Efekti)', test_sfx)

---
## Final Rapor

In [ ]:
print('=' * 80)
print(f'TEST RAPORU - {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('=' * 80)
print(f'{"Test":<35} {"Durum":<8} {"Sure":<8} {"VRAM":<10} {"Boyut":<8}')
print('-' * 80)
pass_ = 0; fail_ = 0
for r in results_log:
    s = r['status']
    if s == 'OK': pass_ += 1; status = 'OK'
    else: fail_ += 1; status = 'FAIL'
    print(f'{r["name"]:<35} {status:<8} {r["duration_s"]:<8}s {r["vram_used_gb"]:<10}GB {r["file_size_mb"]:<8}MB')
    if r['error']:
        print(f'  HATA: {r["error"]}')
print('-' * 80)
print(f'Toplam: {len(results_log)} test | {pass_} basarili | {fail_} basarisiz')
print(f'Ciktilar: {DRIVE_OUTPUT_DIR}')

# Raporu kaydet
report = {'test_date': datetime.now().isoformat(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A', 'results': results_log}
with open(f'{OUTPUT_DIR}/test_report.json', 'w') as f:
    json.dump(report, f, indent=2)
import shutil
shutil.copy2(f'{OUTPUT_DIR}/test_report.json', f'{DRIVE_OUTPUT_DIR}/test_report.json')
print(f'Rapor: {DRIVE_OUTPUT_DIR}/test_report.json')
print('=' * 80)